In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
import statsmodels.api as sm
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import RobustScaler

In [2]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [3]:
# Need to choose patient_id from OUS_D3 in response_OUS
data = list(OUS_D3['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D3 with response_OUS
clinical_train = pd.merge(OUS_D3, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [4]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

## Test dataset: MAASTRO 

In [5]:
(MAASTRO_D3['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [6]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [7]:
# need to choose patient_id from MAASTRO_D3 in response_MAASTRO
data = list(MAASTRO_D3['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 
response_MAASTRO

,patient_id,OS,OS_event,LRC,LRC_event,DFS,DFS_event
0,1,62.43,0.0,62.43,0.0,62.43,0.0
1,2,60.00,0.0,60.00,0.0,60.00,0.0
2,3,44.43,1.0,8.83,1.0,8.83,1.0
3,4,37.20,1.0,19.37,0.0,19.73,1.0
5,6,59.23,0.0,59.23,0.0,59.23,0.0
...,...,...,...,...,...,...,...
109,110,19.00,1.0,13.27,1.0,13.27,1.0
110,111,85.87,1.0,82.83,0.0,85.87,1.0
111,112,42.87,0.0,42.87,0.0,42.87,0.0
112,113,58.93,0.0,58.93,0.0,58.93,0.0


In [8]:
# Merge MAASTRO_D3 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D3, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]
clinical_test

,patient_id,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event
0,1,55,0,0,1,0,0,1,1,1,...,0.028808,0.837387,0.000026,0.001705,0.122209,0.000026,0.008291,0.000341,62.43,0.0
1,2,55,0,0,1,0,0,0,0,0,...,0.049615,0.806842,0.000167,0.002958,0.128976,0.000167,0.008148,0.000335,60.00,0.0
2,3,55,0,0,1,0,0,0,0,1,...,0.019514,0.830272,0.000057,0.001831,0.137282,0.000000,0.008641,0.000229,44.43,1.0
3,4,61,1,0,0,0,1,1,0,1,...,0.047155,0.760949,0.000000,0.003597,0.171595,0.000080,0.013187,0.000240,37.20,1.0
4,6,70,0,0,1,0,0,1,1,1,...,0.033990,0.824865,0.000000,0.002116,0.128134,0.000035,0.009379,0.000529,59.23,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,110,66,1,0,0,0,1,0,0,0,...,0.017489,0.834590,0.000000,0.001321,0.137194,0.000078,0.008706,0.000078,19.00,1.0
95,111,63,0,0,0,0,1,0,0,1,...,0.029979,0.821431,0.000000,0.000543,0.137620,0.000054,0.008907,0.000489,85.87,1.0
96,112,63,0,0,1,0,0,1,1,1,...,0.017354,0.859957,0.000000,0.000988,0.115099,0.000028,0.005700,0.000141,42.87,0.0
97,113,54,0,0,1,0,0,1,1,0,...,0.025399,0.847908,0.000000,0.001487,0.117654,0.000000,0.005759,0.000038,58.93,0.0


In [9]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [10]:
# Some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,...,LBP_021_PET,LBP_030_PET,LBP_102_PET,LBP_111_PET,LBP_120_PET,LBP_201_PET,LBP_210_PET,LBP_300_PET,OS,OS_event


In [11]:
# Set X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# Set y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [12]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [13]:
# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

X_train:  (139, 388)
y_train:  (139,)


In [14]:
# Change the name of a column 'OS_event' in the clincial_test 
clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

In [15]:
# Set X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# Set y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]

# Change y_MAASTRO into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

(99, 390)

## Feature Selection

### COX PLSR 

In [16]:
# Choose features from the result of Cox PLSR in R
plsr = [
"uicc8_III-IV",
"cavum_oris",
"hpv_related",
"charlson",
"shape_MajorAxisLength",
"shape_Elongation",
"hypopharynx"
] 

In [17]:
X_plsr = X.loc[:, plsr]
X_new = X_plsr.copy()

In [18]:
# Selecct the columns from X_MAASTRO
MAASTRO_new = X_MAASTRO.loc[:, plsr]

# Standardization

In [19]:
# Copy the original X for later 
original_X = X.copy()

In [20]:
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Standardize X_new, the new data with the selected features only 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = RobustScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [21]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[X_new.columns]

In [22]:
X_new

,uicc8_III-IV,cavum_oris,hpv_related,charlson,shape_MajorAxisLength,shape_Elongation,hypopharynx
0,0.0,0,0.0,0,42.073251,0.600926,0
1,0.0,0,0.0,1,24.613845,0.841579,0
2,1.0,1,0.0,1,48.030294,0.772821,0
3,0.0,0,0.0,1,25.589900,0.847727,0
4,0.0,0,0.0,1,34.684750,0.831483,0
...,...,...,...,...,...,...,...
134,0.0,0,1.0,0,33.069705,0.680294,0
135,1.0,0,1.0,0,41.043692,0.758193,0
136,0.0,0,1.0,1,36.618802,0.770113,0
137,1.0,0,1.0,1,45.870392,0.628897,0


In [23]:
X_new_std

,uicc8_III-IV,cavum_oris,hpv_related,charlson,shape_MajorAxisLength,shape_Elongation,hypopharynx
0,0.0,0,0.0,0,0.059912,-0.485459,0
1,0.0,0,0.0,1,-0.755402,0.666232,0
2,1.0,1,0.0,1,0.338093,0.337179,0
3,0.0,0,0.0,1,-0.709822,0.695653,0
4,0.0,0,0.0,1,-0.285114,0.617917,0
...,...,...,...,...,...,...,...
134,0.0,0,1.0,0,-0.360532,-0.105628,0
135,1.0,0,1.0,0,0.011834,0.267171,0
136,0.0,0,1.0,1,-0.194798,0.324219,0
137,1.0,0,1.0,1,0.237230,-0.351598,0


In [24]:
MAASTRO_new 

,uicc8_III-IV,cavum_oris,hpv_related,charlson,shape_MajorAxisLength,shape_Elongation,hypopharynx
0,0,0,1,1,50.002093,0.765178,0
1,1,0,0,0,41.753334,0.776540,0
2,1,0,0,1,44.375483,0.697164,0
3,1,0,0,1,46.115989,0.574636,0
4,0,0,1,1,54.394967,0.633419,0
...,...,...,...,...,...,...,...
94,1,0,0,0,34.218615,0.882411,0
95,1,0,0,1,51.046869,0.535802,0
96,1,0,1,1,50.417953,0.716610,0
97,0,0,1,0,44.901412,0.665145,0


In [25]:
MAASTRO_new_std

,uicc8_III-IV,cavum_oris,hpv_related,charlson,shape_MajorAxisLength,shape_Elongation,hypopharynx
0,0,0,1,1,0.430171,0.300600,0
1,1,0,0,0,0.044973,0.354974,0
2,1,0,0,1,0.167421,-0.024893,0
3,1,0,0,1,0.248699,-0.611274,0
4,0,0,1,1,0.635308,-0.329957,0
...,...,...,...,...,...,...,...
94,1,0,0,0,-0.306881,0.861639,0
95,1,0,0,1,0.478960,-0.797117,0
96,1,0,1,1,0.449591,0.068172,0
97,0,0,1,0,0.191981,-0.178126,0


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [26]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:46:42,382] A new study created in memory with name: no-name-93769e59-ea6e-47a3-bf51-1d23e769ddae


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-17 13:46:44,728] A new study created in memory with name: no-name-31f3a0d1-0432-44c4-8f38-86740c41d83c


Fold 1 C-index: 0.7272727272727273
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8431372549019608
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:44,716] Trial 0 finished with value: 0.7856995180201451 and parameters: {}. Best is trial 0 with value: 0.7856995180201451.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7856995180201451], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 42, 424231), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 44, 715132), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7856995180201451


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1621371716412286
Fold 2 IBS: 0.1554762462247702
Fold 3 IBS: 0.12492102173294406
Fold 4 IBS: 0.13624116793254798
Fold 5 IBS: 0.22992325081669146
[I 2024-04-17 13:46:44,876] Trial 0 finished with value: 0.16173977166963643 and parameters: {}. Best is trial 0 with value: 0.16173977166963643.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.16173977166963643], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 44, 748454), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 44, 876091), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.16173977166963643


In [27]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [28]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.786
train_ibs:  0.162


#### Test

In [29]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [30]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.614
IBS score: 0.251


In [31]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [32]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge

#### Train

In [33]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:46:44,966] A new study created in memory with name: no-name-25dfb8a6-7c0a-4ca0-8791-cfb7596b7e0d


  0%|          | 0/1 [00:00<?, ?it/s]

[I 2024-04-17 13:46:45,030] A new study created in memory with name: no-name-645eee19-66d2-4123-8ecf-9156d195cd9e


Fold 1 C-index: 0.5800865800865801
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.7549019607843137
Fold 4 C-index: 0.7257383966244726
Fold 5 C-index: 0.7112676056338029
[I 2024-04-17 13:46:45,028] Trial 0 finished with value: 0.7160060514829768 and parameters: {}. Best is trial 0 with value: 0.7160060514829768.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7160060514829768], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 44, 980216), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 45, 28572), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7160060514829768


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651990447034
Fold 2 IBS: 0.22157790861385837
Fold 3 IBS: 0.20453594300033467
Fold 4 IBS: 0.22473803563950742
Fold 5 IBS: 0.21812430988324932
[I 2024-04-17 13:46:45,126] Trial 0 finished with value: 0.21659054340828404 and parameters: {}. Best is trial 0 with value: 0.21659054340828404.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.21659054340828404], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 45, 47652), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 45, 125985), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.21659054340828404


In [34]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [35]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.716
train_ibs:  0.217


#### Test

In [36]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [37]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.611


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [38]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [39]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:46:45,204] A new study created in memory with name: no-name-c84a86ad-1dcd-4a09-86a2-8899d26e3d3f


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8214285714285714
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815


[I 2024-04-17 13:46:45,364] A new study created in memory with name: no-name-9affe03c-8b4f-47a2-8a94-05f765e22fa2


Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:45,362] Trial 0 finished with value: 0.7831355378826356 and parameters: {}. Best is trial 0 with value: 0.7831355378826356.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7831355378826356], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 45, 223213), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 45, 362544), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7831355378826356


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.16292158130041123
Fold 2 IBS: 0.15660891622574832
Fold 3 IBS: 0.12530869869579714
Fold 4 IBS: 0.13550835255873864
Fold 5 IBS: 0.2298637021937879
[I 2024-04-17 13:46:45,515] Trial 0 finished with value: 0.16204225019489665 and parameters: {}. Best is trial 0 with value: 0.16204225019489665.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.16204225019489665], datetime_start=datetime.datetime(2024, 4, 17, 13, 46, 45, 380484), datetime_complete=datetime.datetime(2024, 4, 17, 13, 46, 45, 515244), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.16204225019489665


In [40]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [41]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.783
train_ibs:  0.162


#### Test

In [42]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [43]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.619


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.248


In [44]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [45]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:46:45,611] A new study created in memory with name: no-name-5aea4676-d26b-492a-8e3f-92209cb21c1f


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:45,771] Trial 0 finished with value: 0.7849212521683497 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7849212521683497.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:45,907] Trial 1 finished with value: 0.7849212521683497 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.7849212521683497.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:46,043] Trial 2 finished with value: 0.7849212521683497 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:49,182] Trial 24 finished with value: 0.7867945014680696 and parameters: {'l1_ratio': 0.1170295107004102}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:49,331] Trial 25 finished with value: 0.7859016443252125 and parameters: {'l1_ratio': 0.12355427757000426}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:49,486] Trial 26 finished with value: 0.7867945014680696 and parameters: {'l1_ratio': 0.11881900182625091}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index:

Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:52,577] Trial 48 finished with value: 0.7849212521683497 and parameters: {'l1_ratio': 0.37534590273406454}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8348214285714286
Fold 3 C-index: 0.8529411764705882
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:52,743] Trial 49 finished with value: 0.7867945014680696 and parameters: {'l1_ratio': 0.06313647075764142}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.8303571428571429
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7089201877934272
[I 2024-04-17 13:46:52,916] Trial 50 finished with value: 0.7849212521683497 and parameters: {'l1_ratio': 0.31297686520002327}. Best is trial 24 with value: 0.7867945014680696.
Fold 1 C-index

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.16285644648646216
Fold 2 IBS: 0.15585311565238952
Fold 3 IBS: 0.12511836909907842
Fold 4 IBS: 0.1355541847291143
Fold 5 IBS: 0.22928182727591367
[I 2024-04-17 13:47:01,602] Trial 0 finished with value: 0.1617327886485916 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.1617327886485916.
Fold 1 IBS: 0.16267368313562722
Fold 2 IBS: 0.154658326768194
Fold 3 IBS: 0.12477067271879583
Fold 4 IBS: 0.135680354593449
Fold 5 IBS: 0.2283905057414276
[I 2024-04-17 13:47:01,781] Trial 1 finished with value: 0.16123470859149874 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.16123470859149874.
Fold 1 IBS: 0.16262991500323248
Fold 2 IBS: 0.15444309239207518
Fold 3 IBS: 0.12472492724276149
Fold 4 IBS: 0.13574382309167732
Fold 5 IBS: 0.22834896398733434
[I 2024-04-17 13:47:02,027] Trial 2 finished with value: 0.16117814434341615 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 2 with value: 0.16117814434341615

Fold 3 IBS: 0.12484560235349428
Fold 4 IBS: 0.13566093624797182
Fold 5 IBS: 0.22863259234209302
[I 2024-04-17 13:47:07,078] Trial 25 finished with value: 0.1613629666488488 and parameters: {'l1_ratio': 0.3735924863066886}. Best is trial 18 with value: 0.16096873185900962.
Fold 1 IBS: 0.16249693451768032
Fold 2 IBS: 0.15407471575191606
Fold 3 IBS: 0.12462167412724169
Fold 4 IBS: 0.13580216133773948
Fold 5 IBS: 0.22812398693462033
[I 2024-04-17 13:47:07,306] Trial 26 finished with value: 0.16102389453383958 and parameters: {'l1_ratio': 0.09964738096662551}. Best is trial 18 with value: 0.16096873185900962.
Fold 1 IBS: 0.16269465518844656
Fold 2 IBS: 0.15471860628416437
Fold 3 IBS: 0.12479095106679061
Fold 4 IBS: 0.13570277321265398
Fold 5 IBS: 0.228515782642803
[I 2024-04-17 13:47:07,532] Trial 27 finished with value: 0.1612845536789717 and parameters: {'l1_ratio': 0.3095665171510701}. Best is trial 18 with value: 0.16096873185900962.
Fold 1 IBS: 0.16261819911919156
Fold 2 IBS: 0.1543484

Fold 4 IBS: 0.1356128361089927
Fold 5 IBS: 0.22900989840060168
[I 2024-04-17 13:47:13,559] Trial 50 finished with value: 0.16157336472602984 and parameters: {'l1_ratio': 0.556147470734502}. Best is trial 30 with value: 0.16096326652500728.
Fold 1 IBS: 0.1624333373693897
Fold 2 IBS: 0.15394957663637424
Fold 3 IBS: 0.12458936001812733
Fold 4 IBS: 0.13582586609568248
Fold 5 IBS: 0.2280410313774311
[I 2024-04-17 13:47:13,814] Trial 51 finished with value: 0.16096783429940092 and parameters: {'l1_ratio': 0.06190359682979441}. Best is trial 30 with value: 0.16096326652500728.
Fold 1 IBS: 0.16253247777451776
Fold 2 IBS: 0.1541462203291175
Fold 3 IBS: 0.12463992403596175
Fold 4 IBS: 0.1357894968359305
Fold 5 IBS: 0.22817219180657222
[I 2024-04-17 13:47:14,041] Trial 52 finished with value: 0.16105606215641993 and parameters: {'l1_ratio': 0.12090276396547626}. Best is trial 30 with value: 0.16096326652500728.
Fold 1 IBS: 0.16241206809075606
Fold 2 IBS: 0.1539383242205832
Fold 3 IBS: 0.124584075

Fold 3 IBS: 0.12461678026567687
Fold 4 IBS: 0.13578865586683128
Fold 5 IBS: 0.2280285692912383
[I 2024-04-17 13:47:19,293] Trial 75 finished with value: 0.16100082036680494 and parameters: {'l1_ratio': 0.08574176873451331}. Best is trial 71 with value: 0.160953763199958.
Fold 1 IBS: 0.16252833558230145
Fold 2 IBS: 0.154183024867559
Fold 3 IBS: 0.12464602153595039
Fold 4 IBS: 0.1357958899958188
Fold 5 IBS: 0.2280907143900187
[I 2024-04-17 13:47:19,597] Trial 76 finished with value: 0.1610487972743297 and parameters: {'l1_ratio': 0.12390315666114743}. Best is trial 71 with value: 0.160953763199958.
Fold 1 IBS: 0.2133842635603898
Fold 2 IBS: 0.15386332158249336
Fold 3 IBS: 0.12456490067016217
Fold 4 IBS: 0.22305840203192154
Fold 5 IBS: 0.22799718809217512
[I 2024-04-17 13:47:19,821] Trial 77 finished with value: 0.18857361518742838 and parameters: {'l1_ratio': 0.029605630293830613}. Best is trial 71 with value: 0.160953763199958.
Fold 1 IBS: 0.1626013379903515
Fold 2 IBS: 0.15433295522417

In [46]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [47]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.787
train_ibs:  0.161


#### Test

In [48]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [49]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.1170295107004102)

test_cindex : 0.617


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.05355583928772211)

test_ibs:  0.247


In [50]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [51]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 13:47:25,139] A new study created in memory with name: no-name-cd514159-06b8-4a9e-898f-33770b1b14e3


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7294372294372294
Fold 2 C-index: 0.7544642857142857
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8059071729957806
Fold 5 C-index: 0.7112676056338029
[I 2024-04-17 13:47:26,797] Trial 0 finished with value: 0.7639407489522982 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7639407489522982.
Fold 1 C-index: 0.7056277056277056
Fold 2 C-index: 0.8125
Fold 3 C-index: 0.8921568627450981
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.7018779342723005
[I 2024-04-17 13:47:29,197] Trial 1 finished with value: 0.7844578169847171 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_features

Fold 1 C-index: 0.6601731601731602
Fold 2 C-index: 0.7901785714285714
Fold 3 C-index: 0.8235294117647058
Fold 4 C-index: 0.7890295358649789
Fold 5 C-index: 0.6924882629107981
[I 2024-04-17 13:50:29,063] Trial 16 finished with value: 0.7510797884284428 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 17, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 498, 'oob_score': True, 'max_samples': 0.17576991206596454, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.0844290597343094, 'warm_start': True}. Best is trial 14 with value: 0.811228519836272.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 13:50:30,205] Trial 17 finished with value: 0.5 and parameters: {'min_samples_split': 20, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 5, 'n_estimators': 431, 'oob_score': True, 'max_samples': 0.378960381234881, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.20515024926941233, 'warm_start': Tru

Fold 1 C-index: 0.7294372294372294
Fold 2 C-index: 0.8705357142857143
Fold 3 C-index: 0.8970588235294118
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.7535211267605634
[I 2024-04-17 13:50:35,269] Trial 31 finished with value: 0.8188869501106006 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 8, 'min_samples_leaf': 13, 'max_depth': 15, 'n_estimators': 192, 'oob_score': True, 'max_samples': 0.8314158504579381, 'max_features': 'log2', 'min_weight_fraction_leaf': 0.1894074705194179, 'warm_start': True}. Best is trial 24 with value: 0.823796809887507.
Fold 1 C-index: 0.7316017316017316
Fold 2 C-index: 0.8660714285714286
Fold 3 C-index: 0.9166666666666666
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.7582159624413145
[I 2024-04-17 13:50:35,613] Trial 32 finished with value: 0.8232875291642451 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 7, 'min_samples_leaf': 11, 'max_depth': 18, 'n_estimators': 87, 'oob_score': True, 'max_samples': 0.7411057479667147, 

...

Fold 5 IBS: 0.20629952789346637
[I 2024-04-17 13:52:00,268] Trial 31 finished with value: 0.1662488401287507 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 10, 'min_samples_leaf': 6, 'max_depth': 20, 'n_estimators': 409, 'oob_score': False, 'max_samples': 0.5937734971360611, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.045073432551245296}. Best is trial 23 with value: 0.16406727501837753.
Fold 1 IBS: 0.17221494847375315
Fold 2 IBS: 0.16332057162458086
Fold 3 IBS: 0.13924621253579658
Fold 4 IBS: 0.13955671627374916
Fold 5 IBS: 0.20848447341582182
[I 2024-04-17 13:52:02,915] Trial 32 finished with value: 0.16456458446474032 and parameters: {'min_samples_split': 3, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 18, 'n_estimators': 422, 'oob_score': False, 'max_samples': 0.5048409346506281, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.041946651269943944}. Best is trial 23 with value: 0.16406727501837753.
Fold 1 IBS: 0.1745623497267235
Fold 2 IBS: 0.1

Fold 1 IBS: 0.1971354741691935
Fold 2 IBS: 0.16636863415022685
Fold 3 IBS: 0.1692145952249857
Fold 4 IBS: 0.1441551060476205
Fold 5 IBS: 0.21183108033751186
[I 2024-04-17 13:52:53,389] Trial 47 finished with value: 0.17774097798590768 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 7, 'min_samples_leaf': 13, 'max_depth': 18, 'n_estimators': 329, 'oob_score': True, 'max_samples': 0.7284286912916192, 'max_features': None, 'min_weight_fraction_leaf': 0.007042646978595906}. Best is trial 46 with value: 0.16210028106730762.
Fold 1 IBS: 0.173257616030805
Fold 2 IBS: 0.16712747227278846
Fold 3 IBS: 0.15059712575175793
Fold 4 IBS: 0.14859926861378026
Fold 5 IBS: 0.20428399666953267
[I 2024-04-17 13:52:55,771] Trial 48 finished with value: 0.16877309586773287 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 3, 'min_samples_leaf': 2, 'max_depth': 17, 'n_estimators': 365, 'oob_score': False, 'max_samples': 0.6356303622551829, 'max_features': 'sqrt', 'min_weight_fraction_leaf'

Fold 5 IBS: 0.2024415952436482
[I 2024-04-17 13:53:42,968] Trial 62 finished with value: 0.16493109295910902 and parameters: {'min_samples_split': 4, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 19, 'n_estimators': 344, 'oob_score': False, 'max_samples': 0.31219984631291253, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.022790585175757945}. Best is trial 46 with value: 0.16210028106730762.
Fold 1 IBS: 0.17181009881095452
Fold 2 IBS: 0.1699080534892515
Fold 3 IBS: 0.15451888340009945
Fold 4 IBS: 0.15244603875471305
Fold 5 IBS: 0.20463195755314145
[I 2024-04-17 13:53:46,003] Trial 63 finished with value: 0.17066300640163198 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 3, 'min_samples_leaf': 3, 'max_depth': 19, 'n_estimators': 327, 'oob_score': False, 'max_samples': 0.373917735738639, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.05547383035537368}. Best is trial 46 with value: 0.16210028106730762.
Fold 1 IBS: 0.17280736029182792
Fold 2 IBS: 0.163

Fold 1 IBS: 0.16919228147273832
Fold 2 IBS: 0.17252035375702687
Fold 3 IBS: 0.15545488892248155
Fold 4 IBS: 0.15513587788347408
Fold 5 IBS: 0.20384083862390687
[I 2024-04-17 13:54:33,733] Trial 78 finished with value: 0.17122884813192554 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 6, 'min_samples_leaf': 5, 'max_depth': 7, 'n_estimators': 246, 'oob_score': False, 'max_samples': 0.3514556475353373, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.06420539756634373}. Best is trial 73 with value: 0.16110184507125572.
Fold 1 IBS: 0.17840279861798333
Fold 2 IBS: 0.19366925370365148
Fold 3 IBS: 0.16628254344058807
Fold 4 IBS: 0.1755191675136859
Fold 5 IBS: 0.20917537027659575
[I 2024-04-17 13:54:36,940] Trial 79 finished with value: 0.1846098267105009 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 10, 'min_samples_leaf': 14, 'max_depth': 3, 'n_estimators': 293, 'oob_score': False, 'max_samples': 0.6586689721518522, 'max_features': 'sqrt', 'min_weight_fraction_

Fold 5 IBS: 0.20635483370375807
[I 2024-04-17 13:55:29,931] Trial 93 finished with value: 0.16290617835025206 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 9, 'min_samples_leaf': 3, 'max_depth': 8, 'n_estimators': 319, 'oob_score': False, 'max_samples': 0.5422942375084822, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.012802544117543141}. Best is trial 73 with value: 0.16110184507125572.
Fold 1 IBS: 0.17176456286813024
Fold 2 IBS: 0.16324995065853418
Fold 3 IBS: 0.1379990506096137
Fold 4 IBS: 0.13820954686984363
Fold 5 IBS: 0.20786497976045762
[I 2024-04-17 13:55:33,259] Trial 94 finished with value: 0.16381761815331589 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 9, 'min_samples_leaf': 4, 'max_depth': 8, 'n_estimators': 309, 'oob_score': False, 'max_samples': 0.538718115810859, 'max_features': 'sqrt', 'min_weight_fraction_leaf': 0.013582457774989606}. Best is trial 73 with value: 0.16110184507125572.
Fold 1 IBS: 0.17101111168768682
Fold 2 IBS: 0.1606

In [52]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [53]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.828
train_ibs:  0.161


#### Test

In [54]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [55]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=16, max_features='auto', max_leaf_nodes=6,
                     max_samples=0.7227199672725352, min_samples_leaf=9,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.10754518872974402,
                     n_estimators=53, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.634


RandomSurvivalForest(max_depth=20, max_leaf_nodes=8,
                     max_samples=0.3940770016576495, min_samples_leaf=1,
                     min_samples_split=5,
                     min_weight_fraction_leaf=0.0005154226962176161,
                     n_estimators=317, random_state=123)

test_ibs:  0.212


In [56]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [57]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [58]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 13:55:53,884] A new study created in memory with name: no-name-abbaae04-c027-4d7f-98f6-1a9d4bb2bbc8


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7532467532467533
Fold 2 C-index: 0.8482142857142857
Fold 3 C-index: 0.8725490196078431
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.7136150234741784
[I 2024-04-17 13:55:55,138] Trial 0 finished with value: 0.8003942147208484 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.8003942147208484.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 13:55:58,459] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7965367965367965
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.8164556962025317
Fold 5 C-index: 0.6784037558685446
[I 2024-04-17 13:56:36,867] Trial 16 finished with value: 0.7894942357159722 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 17, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 499, 'oob_score': False, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.6535355702555379, 'min_weight_fraction_leaf': 0.1906011147608997}. Best is trial 6 with value: 0.8037242254719681.
Fold 1 C-index: 0.79004329004329
Fold 2 C-index: 0.8035714285714286
Fold 3 C-index: 0.8480392156862745
Fold 4 C-index: 0.820675105485232
Fold 5 C-index: 0.6455399061032864
[I 2024-04-17 13:56:38,012] Trial 17 finished with value: 0.7815737891779023 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 10, 'min_samples_leaf': 8, 'max_depth': 10, 'n_estimators': 384, 'oob_score': False, 'warm_start': True, 'max_features'

Fold 1 C-index: 0.7705627705627706
Fold 2 C-index: 0.8392857142857143
Fold 3 C-index: 0.8725490196078431
Fold 4 C-index: 0.8143459915611815
Fold 5 C-index: 0.6901408450704225
[I 2024-04-17 13:56:52,414] Trial 31 finished with value: 0.7973768682175864 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 6, 'min_samples_leaf': 6, 'max_depth': 9, 'n_estimators': 456, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.43716218120232875, 'min_weight_fraction_leaf': 0.0011988413006669512}. Best is trial 6 with value: 0.8037242254719681.
Fold 1 C-index: 0.7489177489177489
Fold 2 C-index: 0.84375
Fold 3 C-index: 0.8725490196078431
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.6807511737089202
[I 2024-04-17 13:56:53,614] Trial 32 finished with value: 0.7945944323287589 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 3, 'min_samples_leaf': 7, 'max_depth': 13, 'n_estimators': 414, 'oob_score': False, 'warm_start': True, 'max_features': 'log

...

Fold 1 IBS: 0.17403611604271424
Fold 2 IBS: 0.19528122214997157
Fold 3 IBS: 0.1682737691725757
Fold 4 IBS: 0.17431224647317028
Fold 5 IBS: 0.22003391698754424
[I 2024-04-17 14:01:13,298] Trial 30 finished with value: 0.18638745416519523 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 8, 'n_estimators': 457, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.3838693489032564, 'min_weight_fraction_leaf': 0.03868084640768137}. Best is trial 12 with value: 0.17446544415091259.
Fold 1 IBS: 0.16740215580950432
Fold 2 IBS: 0.1870998484969457
Fold 3 IBS: 0.1565756547463698
Fold 4 IBS: 0.15710585664080282
Fold 5 IBS: 0.22435467702622547
[I 2024-04-17 14:01:18,280] Trial 31 finished with value: 0.17850763854396962 and parameters: {'min_samples_split': 13, 'max_leaf_nodes': 6, 'min_samples_leaf': 3, 'max_depth': 10, 'n_estimators': 409, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.1576705560383145
Fold 2 IBS: 0.1822547945562819
Fold 3 IBS: 0.145232345938588
Fold 4 IBS: 0.14600743348144415
Fold 5 IBS: 0.225516129704015
[I 2024-04-17 14:02:15,643] Trial 45 finished with value: 0.17133625194372873 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 8, 'n_estimators': 234, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.43514597742187, 'min_weight_fraction_leaf': 0.002921383171058025}. Best is trial 45 with value: 0.17133625194372873.
Fold 1 IBS: 0.16543939753834064
Fold 2 IBS: 0.18388164982951516
Fold 3 IBS: 0.15627011252742087
Fold 4 IBS: 0.15694020536835582
Fold 5 IBS: 0.2276949726422081
[I 2024-04-17 14:02:19,603] Trial 46 finished with value: 0.17804526758116812 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 175, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.4288

Fold 1 IBS: 0.18119331875762684
Fold 2 IBS: 0.19945961417160082
Fold 3 IBS: 0.17386579559292806
Fold 4 IBS: 0.18335508271677708
Fold 5 IBS: 0.21758328735458493
[I 2024-04-17 14:03:25,316] Trial 60 finished with value: 0.19109141971870353 and parameters: {'min_samples_split': 14, 'max_leaf_nodes': 11, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 248, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.19892992997964112, 'min_weight_fraction_leaf': 0.06728588768996817}. Best is trial 54 with value: 0.16937071890429603.
Fold 1 IBS: 0.1614351110786523
Fold 2 IBS: 0.1826286159456077
Fold 3 IBS: 0.1514993289026967
Fold 4 IBS: 0.15264281381011963
Fold 5 IBS: 0.2249499445361135
[I 2024-04-17 14:03:29,652] Trial 61 finished with value: 0.17463116285463795 and parameters: {'min_samples_split': 9, 'max_leaf_nodes': 9, 'min_samples_leaf': 1, 'max_depth': 7, 'n_estimators': 222, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0

Fold 1 IBS: 0.1710934246390583
Fold 2 IBS: 0.18514706849496712
Fold 3 IBS: 0.15982746349670793
Fold 4 IBS: 0.16688100680245713
Fold 5 IBS: 0.21734234000579852
[I 2024-04-17 14:04:47,052] Trial 75 finished with value: 0.1800582606877978 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 9, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 203, 'oob_score': False, 'warm_start': False, 'max_features': 1, 'max_samples': 0.5118766388173772, 'min_weight_fraction_leaf': 0.028141334637192594}. Best is trial 54 with value: 0.16937071890429603.
Fold 1 IBS: 0.16515141877161646
Fold 2 IBS: 0.1850313180620979
Fold 3 IBS: 0.15552022048760814
Fold 4 IBS: 0.15702862487164565
Fold 5 IBS: 0.22563306053778293
[I 2024-04-17 14:04:53,540] Trial 76 finished with value: 0.17767292854615024 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 10, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 280, 'oob_score': False, 'warm_start': False, 'max_features': 'log2', 'max_samples': 0.4800

Fold 1 IBS: 0.16516760895709928
Fold 2 IBS: 0.1844083930646581
Fold 3 IBS: 0.13378241933973636
Fold 4 IBS: 0.13815322880559427
Fold 5 IBS: 0.22955692785142726
[I 2024-04-17 14:06:09,452] Trial 90 finished with value: 0.17021371560370308 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 7, 'n_estimators': 262, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.41453164831674366, 'min_weight_fraction_leaf': 0.029911077163684654}. Best is trial 54 with value: 0.16937071890429603.
Fold 1 IBS: 0.1653205855276024
Fold 2 IBS: 0.18394175946551414
Fold 3 IBS: 0.1340975511411521
Fold 4 IBS: 0.13821860324428945
Fold 5 IBS: 0.22996106804044042
[I 2024-04-17 14:06:14,325] Trial 91 finished with value: 0.1703079134837997 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 12, 'min_samples_leaf': 2, 'max_depth': 20, 'n_estimators': 257, 'oob_score': False, 'warm_start': False, 'max_features': None, 'max_samples': 0.4

In [59]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [60]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.829
train_ibs:  0.167


#### Test

In [61]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [62]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=18, max_features=None, max_leaf_nodes=11,
                   max_samples=0.9786700473362971, min_samples_leaf=2,
                   min_samples_split=13,
                   min_weight_fraction_leaf=0.03145482687545106,
                   n_estimators=186, oob_score=True, random_state=123,
                   warm_start=True)

C-index score: 0.62


ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=13,
                   max_samples=0.4593899319291545, min_samples_leaf=1,
                   min_weight_fraction_leaf=0.013192089384711253,
                   n_estimators=194, random_state=123)

IBS: 0.244


In [63]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis


#### Train

In [64]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-17 14:06:54,752] A new study created in memory with name: no-name-7beded0d-f3fc-4494-9c7f-3d9a4a841612


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:07:31,434] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:07:59,208] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:20:17,285] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7591992996245486.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:22:01,521] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-17 14:36:52,710] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7591992996245486.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.6103286384976526
[I 2024-04-17 14:38:29,198] Trial 26 finished with value: 0.5772199484787512 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_es

...

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609557
[I 2024-04-17 16:11:52,300] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9992870370700113, 'learning_rate': 0.022847552015173876, 'dropout_rate': 0.1556807870961761, 'n_estimators': 451, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.1403134453903068, 'max_features': 'auto', 'min_impurity_decrease': 2.7552659293421345e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 17, 'max_leaf_nodes': 14, 'min_samples_leaf': 10, 'max_depth': 2}. Best is trial 42 with value: 0.21330979403976286.
Fold 1 IBS: 0.2125645258346338
Fold 2 IBS: 0.22018161964251412
Fold 3 IBS: 0.2031089706247819
Fold 4 IBS: 0.22244912786929444
Fold 5 IBS: 0.21730858129617295
[I 2024-04-17 16:12:33,630] Trial 45 finished with value: 0.21512256505347946 and parameters: {'subsample': 0.8888212863898438, 'learning_rate': 0.01524983211

Fold 3 IBS: 0.20287635124596737
Fold 4 IBS: 0.22157930195318806
Fold 5 IBS: 0.21757113828505154
[I 2024-04-17 16:23:30,289] Trial 55 finished with value: 0.21482590116759398 and parameters: {'subsample': 0.9703353679292269, 'learning_rate': 0.023646082998228058, 'dropout_rate': 0.2676210325613897, 'n_estimators': 481, 'criterion': 'squared_error', 'ccp_alpha': 0.036860238643527846, 'min_weight_fraction_leaf': 0.21798842867076448, 'max_features': None, 'min_impurity_decrease': 4.086647023052284e-07, 'validation_fraction': 0.8868940629916056, 'min_samples_split': 18, 'max_leaf_nodes': 15, 'min_samples_leaf': 10, 'max_depth': 4}. Best is trial 42 with value: 0.21330979403976286.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-17 16:24:47,535] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.9556274374724505, 'learning_rate': 0.022777236

Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:29:03,508] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9699873838014033, 'learning_rate': 0.012103946738880062, 'dropout_rate': 0.14222380728521514, 'n_estimators': 115, 'criterion': 'squared_error', 'ccp_alpha': 0.29393333981111347, 'min_weight_fraction_leaf': 0.030567908908384282, 'max_features': 'sqrt', 'min_impurity_decrease': 1.094611645458025e-07, 'validation_fraction': 0.8187342892372662, 'min_samples_split': 17, 'max_leaf_nodes': 12, 'min_samples_leaf': 10, 'max_depth': 12}. Best is trial 42 with value: 0.21330979403976286.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609563
[I 2024-04-17 16:29:04,599] Trial 68 finished with value: 0.21659054862241586 and paramete

Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:29:38,821] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9362147727385395, 'learning_rate': 0.02736082926046323, 'dropout_rate': 0.26541734467806316, 'n_estimators': 77, 'criterion': 'friedman_mse', 'ccp_alpha': 0.2910342652485194, 'min_weight_fraction_leaf': 0.22600697495173297, 'max_features': 'log2', 'min_impurity_decrease': 2.73345967920419e-07, 'validation_fraction': 0.9235701874070005, 'min_samples_split': 16, 'max_leaf_nodes': 17, 'min_samples_leaf': 11, 'max_depth': 4}. Best is trial 72 with value: 0.21316663318669074.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609574
[I 2024-04-17 16:29:46,158] Trial 79 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.9100835469049136, '

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-17 16:30:50,562] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8535484633772966, 'learning_rate': 0.03949586140355399, 'dropout_rate': 0.26821996041255114, 'n_estimators': 41, 'criterion': 'squared_error', 'ccp_alpha': 1.1917037663435968, 'min_weight_fraction_leaf': 0.17574893192422542, 'max_features': 'log2', 'min_impurity_decrease': 4.344142999314639e-07, 'validation_fraction': 0.8624882565770394, 'min_samples_split': 19, 'max_leaf_nodes': 16, 'min_samples_leaf': 14, 'max_depth': 6}. Best is trial 72 with value: 0.21316663318669074.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-17 16:30:56,521] Trial 90 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.7850349664677921, 'learning_rate': 0.0249799521

In [65]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [66]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.799
train_ibs:  0.213


#### Test

In [67]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [68]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.008819927083844869,
                                 criterion='squared_error',
                                 dropout_rate=0.6757649295445344,
                                 learning_rate=0.023239756862861477,
                                 max_depth=1, max_features='sqrt',
                                 max_leaf_nodes=16,
                                 min_impurity_decrease=0.0021237516252394237,
                                 min_samples_leaf=18, min_samples_split=17,
                                 min_weight_fraction_leaf=0.1942611713172168,
                                 n_estimators=53, random_state=123,
                                 subsample=0.6001028949487068,
                                 validation_fraction=0.7847712019774569)

C-index score: 0.638


GradientBoostingSurvivalAnalysis(ccp_alpha=0.004791400710493756,
                                 dropout_rate=0.2700967141743007,
                                 learning_rate=0.04131861156773751, max_depth=6,
                                 max_features=0.1, max_leaf_nodes=17,
                                 min_impurity_decrease=2.7153758955492077e-07,
                                 min_samples_leaf=14, min_samples_split=18,
                                 min_weight_fraction_leaf=0.155227755776175,
                                 n_estimators=87, random_state=123,
                                 subsample=0.9163996694308585,
                                 validation_fraction=0.9936760144532607)

IBS: 0.22


In [69]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [70]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

In [71]:
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Robust Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = RobustScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-17 16:34:07,073] A new study created in memory with name: no-name-0a6fedd0-0381-431e-90e3-39c295d6a36c


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:34:08,482] Trial 0 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7633928571428571
Fold 3 C-index: 0.7156862745098039
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:34:19,655] Trial 1 finished with value: 0.6818827456832424 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6818827456832424.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7156862745098039
Fold 

Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.8080357142857143
Fold 3 C-index: 0.7303921568627451
Fold 4 C-index: 0.70042194092827
Fold 5 C-index: 0.6995305164319249
[I 2024-04-17 16:35:49,648] Trial 19 finished with value: 0.7010959791216443 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.7254192287154788, 'n_estimators': 117, 'learning_rate': 0.09614402133777997}. Best is trial 12 with value: 0.7134956220774182.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.676056338028169
[I 2024-04-17 16:35:59,613] Trial 20 finished with value: 0.6882663906905411 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.18040218016888274, 'n_estimators': 423, 'learning_rate': 0.07788582119853761}. Best is trial 12 with value: 0.7134956220774182.
Fold 1 C-index: 0.5757575757575758
Fold 2 C-index: 0.7946428571428571
Fold 3 C-index: 0.7156862745098039
Fo

Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.78125
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6713615023474179
[I 2024-04-17 16:37:27,320] Trial 38 finished with value: 0.6864345664115337 and parameters: {'subsample': 0.24127903257141559, 'dropout_rate': 0.37281071596569393, 'n_estimators': 189, 'learning_rate': 0.08285381104305417}. Best is trial 12 with value: 0.7134956220774182.
Fold 1 C-index: 0.5627705627705628
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7205882352941176
Fold 4 C-index: 0.6962025316455697
Fold 5 C-index: 0.6666666666666666
[I 2024-04-17 16:37:32,379] Trial 39 finished with value: 0.6863884564182405 and parameters: {'subsample': 0.314034710745663, 'dropout_rate': 0.13241076213324454, 'n_estimators': 287, 'learning_rate': 0.06892741183938003}. Best is trial 12 with value: 0.7134956220774182.
Fold 1 C-index: 0.5670995670995671
Fold 2 C-index: 0.7857142857142857
Fold 3 C-index: 0.7303921568627451
Fold 4 C-i

...

Fold 2 IBS: 0.17864957876196913
Fold 3 IBS: 0.26166561006271827
Fold 4 IBS: 0.2201305954349429
Fold 5 IBS: 0.21615036620181424
[I 2024-04-17 16:45:24,462] Trial 19 finished with value: 0.23678118325944894 and parameters: {'subsample': 0.38793667853472646, 'dropout_rate': 0.22827203020188386, 'n_estimators': 324, 'learning_rate': 0.08697733590757546}. Best is trial 6 with value: 0.17670464524230747.
Fold 1 IBS: 0.21460957699186584
Fold 2 IBS: 0.15203436467055678
Fold 3 IBS: 0.17224234724087079
Fold 4 IBS: 0.17149898083734882
Fold 5 IBS: 0.17345777825397396
[I 2024-04-17 16:45:26,075] Trial 20 finished with value: 0.17676860959892324 and parameters: {'subsample': 0.6102628884762309, 'dropout_rate': 0.5558690358940764, 'n_estimators': 149, 'learning_rate': 0.017331004877487094}. Best is trial 6 with value: 0.17670464524230747.
Fold 1 IBS: 0.21318490797122427
Fold 2 IBS: 0.15475321350865665
Fold 3 IBS: 0.1731329507408618
Fold 4 IBS: 0.17295939262836466
Fold 5 IBS: 0.17437814582988778
[I 20

Fold 2 IBS: 0.1816061154024669
Fold 3 IBS: 0.2540361428973788
Fold 4 IBS: 0.20902566039470935
Fold 5 IBS: 0.21104928161932168
[I 2024-04-17 16:46:45,226] Trial 38 finished with value: 0.23014443567509324 and parameters: {'subsample': 0.525489298855873, 'dropout_rate': 0.7592515401114239, 'n_estimators': 492, 'learning_rate': 0.04117059612157671}. Best is trial 29 with value: 0.1755106652924051.
Fold 1 IBS: 0.26527484844671456
Fold 2 IBS: 0.15047213618254998
Fold 3 IBS: 0.21754707587700448
Fold 4 IBS: 0.18073341212504015
Fold 5 IBS: 0.19187243117143413
[I 2024-04-17 16:46:49,868] Trial 39 finished with value: 0.20117998076054863 and parameters: {'subsample': 0.7558079539823906, 'dropout_rate': 0.8890898259545601, 'n_estimators': 353, 'learning_rate': 0.028863709537211436}. Best is trial 29 with value: 0.1755106652924051.
Fold 1 IBS: 0.229163612652874
Fold 2 IBS: 0.14092389806545633
Fold 3 IBS: 0.1731926585867215
Fold 4 IBS: 0.16527603623773815
Fold 5 IBS: 0.17221215834324385
[I 2024-04-

Fold 3 IBS: 0.17509139432661858
Fold 4 IBS: 0.16496597424322587
Fold 5 IBS: 0.17289856346959426
[I 2024-04-17 16:47:36,095] Trial 57 finished with value: 0.1769653766756531 and parameters: {'subsample': 0.96492509495326, 'dropout_rate': 0.8251715381087623, 'n_estimators': 175, 'learning_rate': 0.02458639392990227}. Best is trial 29 with value: 0.1755106652924051.
Fold 1 IBS: 0.20567506841359517
Fold 2 IBS: 0.1754507394590557
Fold 3 IBS: 0.1807971692927999
Fold 4 IBS: 0.18821099956298334
Fold 5 IBS: 0.18538568611262302
[I 2024-04-17 16:47:38,834] Trial 58 finished with value: 0.18710393256821142 and parameters: {'subsample': 0.8995542733815092, 'dropout_rate': 0.8547461907712806, 'n_estimators': 233, 'learning_rate': 0.005999698403657619}. Best is trial 29 with value: 0.1755106652924051.
Fold 1 IBS: 0.23409900493697242
Fold 2 IBS: 0.13964613861471284
Fold 3 IBS: 0.1766724220851995
Fold 4 IBS: 0.16499514157855344
Fold 5 IBS: 0.17362095292993807
[I 2024-04-17 16:47:42,495] Trial 59 finish

Fold 3 IBS: 0.1869284710217294
Fold 4 IBS: 0.1989285447631815
Fold 5 IBS: 0.19451667349807425
[I 2024-04-17 16:49:17,155] Trial 76 finished with value: 0.1950120229022327 and parameters: {'subsample': 0.5015222453077708, 'dropout_rate': 0.5299478773071792, 'n_estimators': 341, 'learning_rate': 0.0027779562936136393}. Best is trial 74 with value: 0.175186319170446.
Fold 1 IBS: 0.2553470639685611
Fold 2 IBS: 0.14008958962779353
Fold 3 IBS: 0.19022108651736022
Fold 4 IBS: 0.17026980953216966
Fold 5 IBS: 0.19279670916022976
[I 2024-04-17 16:49:25,660] Trial 77 finished with value: 0.18974485176122285 and parameters: {'subsample': 0.13666979131481322, 'dropout_rate': 0.6722323033044297, 'n_estimators': 459, 'learning_rate': 0.019440814750300062}. Best is trial 74 with value: 0.175186319170446.
Fold 1 IBS: 0.21683329566249418
Fold 2 IBS: 0.15072164418999306
Fold 3 IBS: 0.17055564345481652
Fold 4 IBS: 0.1703049714714073
Fold 5 IBS: 0.1750120026120774
[I 2024-04-17 16:49:33,088] Trial 78 finis

Fold 3 IBS: 0.17650989370961295
Fold 4 IBS: 0.1649768179384022
Fold 5 IBS: 0.17358752615653894
[I 2024-04-17 16:51:22,264] Trial 95 finished with value: 0.17754888377091613 and parameters: {'subsample': 0.49260763697644044, 'dropout_rate': 0.5264582292092028, 'n_estimators': 352, 'learning_rate': 0.013128187171003648}. Best is trial 74 with value: 0.175186319170446.
Fold 1 IBS: 0.2866906463342198
Fold 2 IBS: 0.1673131983448597
Fold 3 IBS: 0.2449845962777135
Fold 4 IBS: 0.20002389838361384
Fold 5 IBS: 0.2037766350360634
[I 2024-04-17 16:51:27,247] Trial 96 finished with value: 0.22055779487529406 and parameters: {'subsample': 0.308291704410664, 'dropout_rate': 0.4736950461791585, 'n_estimators': 316, 'learning_rate': 0.054487916689644846}. Best is trial 74 with value: 0.175186319170446.
Fold 1 IBS: 0.2641416731850371
Fold 2 IBS: 0.1496045701947549
Fold 3 IBS: 0.2159079716925395
Fold 4 IBS: 0.17958191559776393
Fold 5 IBS: 0.19139670221522273
[I 2024-04-17 16:51:33,425] Trial 97 finished 

In [72]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [73]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.713
train_ibs:  0.175


#### Test

In [74]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [75]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.10707700162921632,
                                              learning_rate=0.09635955935176935,
                                              n_estimators=311,
                                              random_state=123,
                                              subsample=0.11211713718471546)

C-index score: 0.6


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5791496160294716,
                                              learning_rate=0.008905079139999657,
                                              n_estimators=411,
                                              random_state=123,
                                              subsample=0.22537252419144232)

IBS: 0.233


In [76]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [77]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.829,1.0
Randomsurvivalforest,0.828,2.0
GradientBoosting,0.799,3.0
CoxElastic,0.787,4.0
CoxPH,0.786,5.0
CoxLasso,0.783,6.0
CoxRidge,0.716,7.0
ComponentwiseGradientBoosting,0.713,8.0


In [78]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
CoxElastic,0.161,1.5
Randomsurvivalforest,0.161,1.5
CoxPH,0.162,3.5
CoxLasso,0.162,3.5
ExtraSurvivalTrees,0.167,5.0
ComponentwiseGradientBoosting,0.175,6.0
GradientBoosting,0.213,7.0
CoxRidge,0.217,8.0


In [79]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.638,1.0
Randomsurvivalforest,0.634,2.0
ExtraSurvivalTrees,0.620,3.0
CoxLasso,0.619,4.0
CoxElastic,0.617,5.0
CoxPH,0.614,6.0
CoxRidge,0.611,7.0
ComponentwiseGradientBoosting,0.600,8.0


In [80]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs 

,IBS,rank
Randomsurvivalforest,0.212,1.0
GradientBoosting,0.220,2.0
CoxRidge,0.221,3.0
ComponentwiseGradientBoosting,0.233,4.0
ExtraSurvivalTrees,0.244,5.0
CoxElastic,0.247,6.0
CoxLasso,0.248,7.0
CoxPH,0.251,8.0


In [81]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d3/os/robust/plsr/' 

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d3_os_robust_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [82]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-17
